In [0]:
# Databricks notebook source

# ==========================================
# GOLD FINANCE ANALYSIS
# ==========================================

from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    count,
    avg,
    round as spark_round,
    max as spark_max,
    min as spark_min,
    when
)


In [0]:

SILVER_TABLE = "personal.finance.silver_transactions"

print(f"Source table: {SILVER_TABLE}")


In [0]:
df_silver = spark.table(SILVER_TABLE)

print(f"Silver rows: {df_silver.count()}")

display(df_silver.limit(20))


In [0]:
df_monthly_summary = (
    df_silver
    .groupBy(
        "year",
        "month_date",
        "month"
    )
    .agg(
        spark_sum(
            when(
                col("transaction_type") == "INCOME",
                col("amount")
            ).otherwise(0)
        ).alias("total_income"),

        spark_sum(
            when(
                col("transaction_type") == "EXPENSE",
                col("amount")
            ).otherwise(0)
        ).alias("total_expense"),

        spark_sum(
            when(
                col("transaction_type") == "INCOME",
                1
            ).otherwise(0)
        ).alias("income_transaction_count"),

        spark_sum(
            when(
                col("transaction_type") == "EXPENSE",
                1
            ).otherwise(0)
        ).alias("expense_transaction_count"),

        avg(
            when(
                col("transaction_type") == "EXPENSE",
                col("amount")
            )
        ).alias("average_expense")
    )
    .withColumn(
        "savings",
        col("total_income") - col("total_expense")
    )
    .withColumn(
        "savings_percentage",
        when(
            col("total_income") > 0,
            spark_round(
                (col("savings") / col("total_income")) * 100,
                2
            )
        ).otherwise(0)
    )
    .withColumn(
        "total_income",
        spark_round(col("total_income"), 2)
    )
    .withColumn(
        "total_expense",
        spark_round(col("total_expense"), 2)
    )
    .withColumn(
        "savings",
        spark_round(col("savings"), 2)
    )
    .withColumn(
        "average_expense",
        spark_round(col("average_expense"), 2)
    )
    .orderBy("month_date")
)

(
    df_monthly_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("personal.finance.gold_monthly_summary")
)

print("gold_monthly_summary created.")

display(df_monthly_summary)


In [0]:
df_category_summary = (
    df_silver
    .filter(col("transaction_type") == "EXPENSE")
    .groupBy(
        "year",
        "month_date",
        "month",
        "category"
    )
    .agg(
        spark_sum("amount").alias("total_expense"),
        count("*").alias("transaction_count"),
        avg("amount").alias("average_expense"),
        spark_max("amount").alias("maximum_expense"),
        spark_min("amount").alias("minimum_expense")
    )
    .withColumn(
        "total_expense",
        spark_round(col("total_expense"), 2)
    )
    .withColumn(
        "average_expense",
        spark_round(col("average_expense"), 2)
    )
    .withColumn(
        "maximum_expense",
        spark_round(col("maximum_expense"), 2)
    )
    .withColumn(
        "minimum_expense",
        spark_round(col("minimum_expense"), 2)
    )
    .orderBy(
        "month_date",
        col("total_expense").desc()
    )
)

(
    df_category_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("personal.finance.gold_category_summary")
)

print("gold_category_summary created.")

display(df_category_summary)


In [0]:
# 3. Overall Category Analysis

df_category_overall = (
    df_silver
    .filter(col("transaction_type") == "EXPENSE")
    .groupBy("category")
    .agg(
        spark_sum("amount").alias("total_expense"),
        count("*").alias("transaction_count"),
        avg("amount").alias("average_expense")
    )
    .withColumn(
        "total_expense",
        spark_round(col("total_expense"), 2)
    )
    .withColumn(
        "average_expense",
        spark_round(col("average_expense"), 2)
    )
    .orderBy(
        col("total_expense").desc()
    )
)

(
    df_category_overall
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("personal.finance.gold_category_overall")
)

print("gold_category_overall created.")

display(df_category_overall)


In [0]:
# 4. Monthly Category Trend

df_monthly_category = (
    df_silver
    .filter(col("transaction_type") == "EXPENSE")
    .groupBy(
        "year",
        "month_date",
        "month",
        "category"
    )
    .agg(
        spark_sum("amount").alias("category_expense")
    )
)

# Calculate total expense for each month
from pyspark.sql.window import Window

window_month = Window.partitionBy("month_date")

df_monthly_category = (
    df_monthly_category
    .withColumn(
        "total_monthly_expense",
        spark_sum("category_expense").over(window_month)
    )
    .withColumn(
        "expense_percentage",
        spark_round(
            (
                col("category_expense")
                / col("total_monthly_expense")
            ) * 100,
            2
        )
    )
    .withColumn(
        "category_expense",
        spark_round(col("category_expense"), 2)
    )
    .withColumn(
        "total_monthly_expense",
        spark_round(col("total_monthly_expense"), 2)
    )
    .orderBy(
        "month_date",
        col("category_expense").desc()
    )
)

(
    df_monthly_category
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("personal.finance.gold_monthly_category")
)

print("gold_monthly_category created.")

display(df_monthly_category)


In [0]:
# 5. Top Expenses

df_top_expenses = (
    df_silver
    .filter(col("transaction_type") == "EXPENSE")
    .select(
        "transaction_date",
        "amount",
        "category",
        "description",
        "remark",
        "month"
    )
    .orderBy(col("amount").desc())
)

(
    df_top_expenses
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("personal.finance.gold_top_expenses")
)

print("gold_top_expenses created.")

display(df_top_expenses.limit(50))


In [0]:
# 6. Income Summary

df_income_summary = (
    df_silver
    .filter(col("transaction_type") == "INCOME")
    .groupBy(
        "year",
        "month_date",
        "month"
    )
    .agg(
        spark_sum("amount").alias("total_income"),
        count("*").alias("income_transaction_count"),
        avg("amount").alias("average_income"),
        spark_max("amount").alias("maximum_income"),
        spark_min("amount").alias("minimum_income")
    )
    .withColumn(
        "total_income",
        spark_round(col("total_income"), 2)
    )
    .withColumn(
        "average_income",
        spark_round(col("average_income"), 2)
    )
    .withColumn(
        "maximum_income",
        spark_round(col("maximum_income"), 2)
    )
    .withColumn(
        "minimum_income",
        spark_round(col("minimum_income"), 2)
    )
    .orderBy("month_date")
)

(
    df_income_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("personal.finance.gold_income_summary")
)

print("gold_income_summary created.")

display(df_income_summary)


In [0]:
# 7. Yearly Summary

df_yearly_summary = (
    df_silver
    .groupBy("year")
    .agg(
        spark_sum(
            when(
                col("transaction_type") == "INCOME",
                col("amount")
            ).otherwise(0)
        ).alias("total_income"),

        spark_sum(
            when(
                col("transaction_type") == "EXPENSE",
                col("amount")
            ).otherwise(0)
        ).alias("total_expense"),

        count("*").alias("total_transactions")
    )
    .withColumn(
        "savings",
        col("total_income") - col("total_expense")
    )
    .withColumn(
        "savings_percentage",
        when(
            col("total_income") > 0,
            spark_round(
                (col("savings") / col("total_income")) * 100,
                2
            )
        ).otherwise(0)
    )
    .withColumn(
        "total_income",
        spark_round(col("total_income"), 2)
    )
    .withColumn(
        "total_expense",
        spark_round(col("total_expense"), 2)
    )
    .withColumn(
        "savings",
        spark_round(col("savings"), 2)
    )
    .orderBy("year")
)

(
    df_yearly_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("personal.finance.gold_yearly_summary")
)

print("gold_yearly_summary created.")

display(df_yearly_summary)


In [0]:
%sql

CREATE OR REPLACE TABLE personal.finance.gold_category_yearly AS

SELECT
    year,
    category,

    SUM(
        CASE
            WHEN transaction_type = 'EXPENSE'
            THEN amount
            ELSE 0
        END
    ) AS total_expense

FROM personal.finance.silver_transactions

GROUP BY
    year,
    category;

In [0]:
# Final validation

gold_tables = [
    "gold_monthly_summary",
    "gold_category_summary",
    "gold_category_overall",
    "gold_monthly_category",
    "gold_top_expenses",
    "gold_income_summary",
    "gold_yearly_summary",
    "gold_category_yearly"
]

for table_name in gold_tables:
    full_table_name = f"personal.finance.{table_name}"

    print(
        f"{full_table_name}: "
        f"{spark.table(full_table_name).count()} rows"
    )

print("All Gold tables created successfully.")
